# CS 123 — The Spider: a Trick That Walks 🕷️

Lab 6, Part 6. Pupper lies down like a spider, wiggles its legs, **walks around
in that stance**, and wiggles back up to standing. It takes two policies:

1. **The spider clip trick** (Part A): a tracking policy, like `sit_stay_shake`,
   with three clips: `spider_down` (stand -> lie down -> wiggle -> the spider
   stance), `spider_idle` (the stance) and `spider_up` (wiggle -> stand).
2. **The spider walk** (Part B): a walking policy, like your Lab 4 policy, but
   in the spider stance: torso 1 cm off the ground, legs splayed wide. It tracks
   a *reference gait*, a spider trot in place, the way StableGait tracks the trot.

On the robot, `play_trick spider_walk` plays `spider_down`, then hands the legs
to the spider walk; `move`, `turn` and `arc` walk it around; any other motion
switches back to the clip trick, started at `spider_up`.

*Runtime → Change runtime type → a GPU, then run the cells top to bottom.* Each
training run takes 15–30 minutes; Part B can start while Part A trains if you
open a second copy of this notebook.

## 0. Setup

In [ ]:
REPO_URL = "https://github.com/cs123-stanford/pupper-mjlab.git"
BRANCH = "main"

import os

os.environ["MUJOCO_GL"] = "egl"
!nvidia-smi -L
if not os.path.exists("/content/pupper-mjlab"):
  !git clone --branch {BRANCH} --depth 1 {REPO_URL} /content/pupper-mjlab
%cd /content/pupper-mjlab
!pip install -q -e .

# The running kernel can't see an editable install's path hooks; wire src/
# onto the path so imports work without a runtime restart.
import sys

sys.path.insert(0, "/content/pupper-mjlab/src")
for _m in [k for k in list(sys.modules) if k == "mjlab" or k.startswith("mjlab.")]:
  del sys.modules[_m]
import mjlab.envs  # smoke-test

print("mjlab ready")

In [ ]:
WANDB_API_KEY = ""  # paste your key from https://wandb.ai/authorize between the quotes

import wandb

if WANDB_API_KEY:
  os.environ["WANDB_API_KEY"] = WANDB_API_KEY
  wandb.login(key=WANDB_API_KEY, relogin=True)
else:
  wandb.login()  # no key pasted: falls back to the interactive prompt
ENTITY = wandb.Api().default_entity
print("W&B entity:", ENTITY)

**Watch a policy while it trains.** Run the next cell once. Then, before each
training run, `watch_live(...)` starts a viewer in the background and prints its
link: the robot appears with the first checkpoint (about a minute in) and
switches to newer ones as they land. It pauses itself while no one is watching,
so it costs training nothing then.

In [ ]:
import subprocess
import sys
import time
from pathlib import Path

URL_FILE = Path("/content/viewer_url.txt")
LOG_FILE = Path("/content/viewer.log")
PID_FILE = Path("/content/viewer_pid.txt")


def watch_live(task, motion_file=None):
  # Restart the watcher so it latches onto the run started next, not an old one.
  if PID_FILE.exists():
    subprocess.run(["kill", PID_FILE.read_text().strip()], capture_output=True)
  URL_FILE.unlink(missing_ok=True)
  cmd = [sys.executable, "notebooks/watch_live.py", task, "--url-file", str(URL_FILE)]
  if motion_file is not None:  # a trick: show it tracking its own motion
    cmd += ["--motion-file", str(motion_file)]
  proc = subprocess.Popen(cmd, stdout=LOG_FILE.open("w"), stderr=subprocess.STDOUT)
  PID_FILE.write_text(str(proc.pid))
  for _ in range(45):
    if URL_FILE.exists():
      print("Watch live here:", URL_FILE.read_text())
      break
    time.sleep(2)
  else:
    print("Viewer didn't come up -- check", LOG_FILE)
  print("The robot appears once the first checkpoint lands (~1 min into training).")

## Part A: the spider clip trick

The Pupper monorepo recorded a spider trick: stand, lie down, splay the legs,
wiggle them, lie still. You cut it into three clips. Watch it first (a public
link; stop the cell when you're done):

In [ ]:
from pathlib import Path

import numpy as np

from mjlab.tasks.tracking.config.pupper import motion_design as md
from mjlab.tasks.tracking.config.pupper.tricks import spider

OUT = Path("/content/tricks")
OUT.mkdir(exist_ok=True)
rec = spider.recorded()
print(f"recorded spider: {rec.duration:.1f} s, {len(rec.joint_pos)} frames")
PREVIEW = rec.save_npz(OUT / "spider_recorded.npz")

In [ ]:
!python -m mjlab.tasks.tracking.config.pupper.play_motion {PREVIEW} --no-physics True --viewer viser --share True

**Your Part 5 helpers.** Paste your `cut` and `hold` from the Sit, Stay, Shake
notebook into the next cell. Then write `reverse`: the motion played backwards.

In [ ]:
def cut(motion: md.Motion, start_s: float, end_s: float) -> md.Motion:
  # YOUR CODE HERE: paste your cut() from Part 5
  raise NotImplementedError("paste your cut() from Part 5")


def hold(motion: md.Motion, at_s: float, seconds: float) -> md.Motion:
  # YOUR CODE HERE: paste your hold() from Part 5
  raise NotImplementedError("paste your hold() from Part 5")


def reverse(motion: md.Motion) -> md.Motion:
  # YOUR CODE HERE (TODO S1): a new md.Motion, same fps, every array's frames in reverse order.
  raise NotImplementedError("TODO S1: reverse")


spider.cut, spider.hold, spider.reverse = cut, hold, reverse
assert spider.check_helpers(), "fix the FAIL lines above"

**The clips (TODO S2).** `spider_down` is the recording up to
`spider.WIGGLE_UNTIL_S` (7 s, still wiggling), then
`spider.settle_into_stance(...)` blends it into the stance the spider walk
starts from. `spider_idle` holds that stance. `spider_up` goes from the stance
back to standing, wiggling on the way: which motion you already have does
exactly that, backwards?

Then `ORDER`, the training sequence: as in Part 5, the policy only learns the
clip changes that appear in it.

In [ ]:
def clips() -> list[md.Clip]:
  # YOUR CODE HERE (TODO S2a): md.Clip(name, motion, loop=..., next=...) for
  # spider_down, spider_idle and spider_up.
  raise NotImplementedError("TODO S2a: clips")


ORDER = [
  # YOUR CODE HERE (TODO S2b)
]

library = md.concat_clips(clips(), order=ORDER)
for key, value in library.report.items():
  print(f"{key:36} {value}")
for c in library.clips:
  print(f"  clip {c['name']:12} frames {c['start']:4}-{c['end']:4} "
        f"({(c['end'] - c['start']) / 50:.1f} s) loop={c['loop']} next={c['next']}")
CLIP_FILE = library.save_npz(OUT / "spider_walk.npz")
print("saved", CLIP_FILE)

Replay it with physics. It **FALLS**: pushing up from the belly needs more than
the plain replay's gains. That is what the trained policy is for.

In [ ]:
!python -m mjlab.tasks.tracking.config.pupper.play_motion {CLIP_FILE}

Upload it, start the live viewer, and train the clip trick (about 25 minutes on a T4-class GPU):

In [ ]:
watch_live("Mjlab-Tracking-Flat-Pupper-v3", motion_file=CLIP_FILE)

In [ ]:
!python -m mjlab.tasks.tracking.config.pupper.upload_motion {CLIP_FILE} --name spider_walk

from mjlab.scripts.train import TrainConfig, launch_training
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg
import mjlab.tasks  # register tasks

TRACK_TASK = "Mjlab-Tracking-Flat-Pupper-v3"
env_cfg = load_env_cfg(TRACK_TASK)
env_cfg.scene.num_envs = 4096
agent_cfg = load_rl_cfg(TRACK_TASK)
agent_cfg.max_iterations = 3000
agent_cfg.logger = "wandb"
agent_cfg.run_name = "spider_walk_clips"
launch_training(TRACK_TASK, TrainConfig(env=env_cfg, agent=agent_cfg, registry_name=f"{ENTITY}/mjlab/spider_walk"))

In [ ]:
runs = sorted(Path("logs/rsl_rl/pupper_tracking").glob("*spider_walk_clips*"), key=lambda p: p.stat().st_mtime)
CHECKPOINT = max(runs[-1].glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
print("checkpoint:", CHECKPOINT)
!python -m mjlab.tasks.tracking.config.pupper.play_motion {CLIP_FILE} --checkpoint {CHECKPOINT} --viewer viser --share True --loops 2

## Part B: the spider walk

A walking policy in the spider stance. As in Lab 4, it tracks a **reference
gait** while it follows the speed command: here a *spider trot in place*. The
diagonal leg pairs take turns stepping, twice a second, in the stance; at zero
command the reference is the still stance. The leg motion comes from the
recorded spider: the front legs fold the knee and swing up and out, the back
legs swing up (`spider` in `mdp/gait_reference.py`). Look at it before you
train on it (gravity off: the robot floats, so you see just the legs):

In [ ]:
!python -m mjlab.tasks.pupper_gait.visualize_reference --task Mjlab-SpiderWalk-Flat-Pupper-v3 --gait spider --frequency 2.0 --viser True --share True

**Your reward weights.** The same terms as Lab 4: start from your StableGait
weights, then think about what is different here:

- `base_height` targets 4.2 cm, not 14.5 cm. The motors can't hold this low,
  splayed stance by themselves (with zero actions it sags onto its belly), so
  holding the body up is something the policy has to learn.
- `abduction_angle` pulls the `_2` joints toward 0. In the spider stance they
  are at ±1.1 rad on purpose. What does a negative weight do here?
- `knee_ground_contact`: in the stance, the knees are a couple of centimetres
  off the ground. Sagging puts them on it.
- Commands are slow (at most 0.15 m/s forward, 0.1 m/s sideways, 0.6 rad/s
  turning): the spider creeps.

In [ ]:
TASK = "Mjlab-SpiderWalk-Flat-Pupper-v3"
REWARD_WEIGHTS = {
  # objectives
  "track_linear_velocity": 0.0,  # follow commanded vx/vy
  "track_yaw_velocity": 0.0,  # follow commanded turn rate
  "upright": 0.0,  # stay upright
  "air_time": 0.0,  # take real steps
  "base_height": 0.0,  # hold the body at 4.2 cm
  "gait_tracking": 0.0,  # match the spider reference gait
  "track_angular_velocity": 0.0,  # roll/pitch-rate stabilizer
  # penalties (negative weights)
  "termination": 0.0,  # falling over
  "orientation_l2": 0.0,  # body tilt
  "lin_vel_z_l2": 0.0,  # vertical bouncing
  "ang_vel_xy_l2": 0.0,  # roll/pitch thrash
  "foot_slip": 0.0,  # planted feet sliding
  "action_rate_l2": 0.0,  # jerky actions
  "joint_torques_l2": 0.0,  # effort
  "joint_acc_l2": 0.0,  # joint acceleration
  "stand_still_pose": 0.0,  # fidgeting at zero command
  "stand_still_joint_velocity": 0.0,  # creeping at zero command
  "knee_ground_contact": 0.0,  # knees on the floor
  "abduction_angle": 0.0,  # legs pulled toward 0 (careful: the spider splays on purpose)
  "self_collision_l": 0.0,  # left legs colliding
  "self_collision_r": 0.0,  # right legs colliding
}
ITERATIONS = 4000

Start the live viewer, then train. Watch it learn: the legs first (the body is held up for the first 500 iterations), then walking.

In [ ]:
watch_live(TASK)

In [ ]:
env_cfg = load_env_cfg(TASK)
for name, weight in REWARD_WEIGHTS.items():
  env_cfg.rewards[name].weight = weight
env_cfg.scene.num_envs = 4096
agent_cfg = load_rl_cfg(TASK)
agent_cfg.max_iterations = ITERATIONS
agent_cfg.logger = "wandb"
agent_cfg.run_name = "spider_walk"
launch_training(TASK, TrainConfig(env=env_cfg, agent=agent_cfg))

The first 500 iterations hold the body in the air (the same gait pretraining as
Lab 4): the legs learn the reference before they carry weight. Watch the result
(stop the cell when you're done):

In [ ]:
from dataclasses import asdict

import viser
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
from mjlab.tasks.registry import load_runner_cls
from mjlab.viewer import ViserPlayViewer

runs = sorted(Path("logs/rsl_rl/pupper_spider_walk").glob("*"), key=lambda p: p.stat().st_mtime)
assert runs, "No spider walk runs yet -- train first."
ckpts = sorted(runs[-1].glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
print("Using checkpoint:", ckpts[-1])

device = "cuda:0"
play_cfg = load_env_cfg(TASK, play=True)
play_cfg.scene.num_envs = 1
play_agent = load_rl_cfg(TASK)
env = RslRlVecEnvWrapper(ManagerBasedRlEnv(cfg=play_cfg, device=device), clip_actions=play_agent.clip_actions)
runner = (load_runner_cls(TASK) or MjlabOnPolicyRunner)(env, asdict(play_agent), device=device)
runner.load(str(ckpts[-1]), load_cfg={"actor": True}, strict=True, map_location=device)

server = viser.ViserServer(label="spider", share=True)
print("Open the viewer here:", server.request_share_url())
ViserPlayViewer(env, runner.get_inference_policy(device=device), viser_server=server).run()

Check: it stays low (the body 3–4 cm up, never on its belly), its legs keep
stepping like the reference, and it goes forward, backward, sideways and turns
when commanded. It will be slower than commanded, and that is fine: the spider
creeps.

## Part C: onto the robot

Show a TA both policies in the viewer first: **new trained policies need a TA's
OK** before their first run on the robot. Then, on the Pupper, fetch both into
one trick:

```bash
python3 robot/get_trick.py <clip-trick-run> --walk <spider-walk-run> --name spider_walk
robot/stop.sh && robot/start.sh
```

and fill in `tricks/spider_walk/trick.yaml` (the lab handout says how).